<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/tools/bedrock_agentcore_browser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amazon Bedrock AgentCore Browser Tools

This notebook demonstrates how to use the [Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) Browser tools with LlamaIndex. The toolkit provides a managed Chrome browser for web automation, content extraction, and human-in-the-loop oversight via live view URLs.

## Prerequisites

- An AWS account with Amazon Bedrock AgentCore access
- Configured AWS credentials (via `~/.aws/credentials`, environment variables, or IAM role)
- Required IAM permissions: `bedrock-agentcore:InvokeBrowser`
- Playwright is installed automatically as a dependency of the toolkit

## Install Dependencies

In [ ]:
%pip install -q llama-index llama-index-tools-aws-bedrock-agentcore llama-index-llms-bedrock-converse

## Setup

Create an `AgentCoreBrowserToolSpec` instance. A browser session is started lazily on first navigation.

In [ ]:
from llama_index.tools.aws_bedrock_agentcore import AgentCoreBrowserToolSpec

tool_spec = AgentCoreBrowserToolSpec(region="us-west-2")
tools = tool_spec.to_tool_list()

print(f"Available tools: {[t.metadata.name for t in tools]}")

## Direct Tool Usage

You can call the tool spec methods directly without an agent. Since Jupyter runs inside an asyncio event loop, use the async methods (prefixed with `a`).

### Navigate and Extract Content

In [ ]:
# Navigate to a page (use async methods inside Jupyter's event loop)
result = await tool_spec.anavigate_browser("https://example.com")
print(result)

# Get current page info
result = await tool_spec.acurrent_webpage()
print(result)

In [ ]:
# Extract text from a specific element
result = await tool_spec.aextract_text(selector="h1")
print(f"Page heading: {result}")

### Links and Elements

In [ ]:
# Extract all hyperlinks
result = await tool_spec.aextract_hyperlinks()
print("Hyperlinks:")
print(result)

# Get elements matching a CSS selector
result = await tool_spec.aget_elements("p")
print("\nParagraph elements:")
print(result)

### Live View URL

Generate a presigned URL that lets a human observe the browser session in real time. This is useful for oversight and debugging.

In [ ]:
live_url = await tool_spec.agenerate_live_view_url()
print(f"Live view URL (expires in 5 minutes):\n{live_url}")

### Human-in-the-Loop Control

Use `take_control()` to pause automation and let a human interact with the browser via the live view URL. Call `release_control()` to hand control back to the agent.

In [ ]:
# Pause automation for human interaction
result = await tool_spec.atake_control()
print(result)

# ... human interacts via the live view URL ...

# Resume automation
result = await tool_spec.arelease_control()
print(result)

## Agent Integration

Combine the browser tools with a `FunctionAgent` and `BedrockConverse` LLM to let the agent autonomously browse the web.

In [ ]:
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent

llm = BedrockConverse(model="us.anthropic.claude-sonnet-4-20250514-v1:0")

agent = FunctionAgent(
    tools=tools,
    llm=llm,
)

### Web Scraping Task

Ask the agent to visit Hacker News and summarize the top posts.

In [ ]:
response = await agent.run(
    "Go to https://news.ycombinator.com/ and tell me the titles of the top 5 posts."
)
print(str(response))

### Content Extraction Task

Ask the agent to visit a page and extract specific information.

In [ ]:
response = await agent.run(
    "Go to https://example.com and extract the page heading and all hyperlinks on the page."
)
print(str(response))

## Cleanup

Always clean up browser sessions when done to release resources.

In [ ]:
await tool_spec.cleanup()
print("Browser sessions cleaned up.")

## References

- [Amazon Bedrock AgentCore Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)
- [llama-index-tools-aws-bedrock-agentcore on PyPI](https://pypi.org/project/llama-index-tools-aws-bedrock-agentcore/)